# **NLP-Based Chatbot**

In [ ]:
import json
import random
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
intents_data = {
    "intents": [
        {
            "tag": "greeting",
            "patterns": ["Hi", "Hello", "Hey"],
            "responses": ["Hello!", "Hi there!", "Hey!"]
        },
        {
            "tag": "goodbye",
            "patterns": ["Bye", "Goodbye", "See you"],
            "responses": ["Bye!", "Take care!", "See you later!"]
        },
        {
            "tag": "thanks",
            "patterns": ["Thanks", "Thank you"],
            "responses": ["You're welcome!", "No problem!"]
        }
    ]
}

with open("intents.json", "w") as f:
    json.dump(intents_data, f, indent=4)

print("intents.json is created ")

intents.json is created 


In [ ]:
with open("intents.json") as file:
    data = json.load(file)

In [ ]:
texts = []
labels = []
responses = {}

for intent in data["intents"]:
    responses[intent["tag"]] = intent["responses"]
    for pattern in intent["patterns"]:
        texts.append(pattern)
        labels.append(intent["tag"])

print(len(texts), len(labels))

8 8


In [ ]:
texts=[]
labels=[]
responses={}
for intent in data["intents"]:
  responses[intent["tag"]]=intent[ "responses"]
  for pattern in intent["patterns"]:
    texts.append(pattern)
    labels.append(intent["tag"])

In [ ]:
tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(texts)

X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X, maxlen=5)

print("X shape:", X.shape)

X shape: (8, 5)


In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

encoder = LabelEncoder()
y = encoder.fit_transform(labels)
y = to_categorical(y)

In [ ]:
index_label = encoder.classes_
print(index_label)

['goodbye' 'greeting' 'thanks']


In [ ]:
model = Sequential()
model.add(Dense(16, activation='relu', input_shape=(5,)))
model.add(Dense(16, activation='relu'))
model.add(Dense(len(index_label), activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 419 (1.64 KB)

 Trainable params: 419 (1.64 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X, y, epochs=200, verbose=0)

In [ ]:
def chatbot_response(user_input):
    seq = tokenizer.texts_to_sequences([user_input])
    pad = pad_sequences(seq, maxlen=5)

    prediction = model.predict(pad)
    intent_index = np.argmax(prediction)
    intent = index_label[intent_index]

    return random.choice(responses[intent])

In [ ]:
print("Chatbot ready!")

while True:
    msg = input("You: ")
    if msg.lower() == "quit":
        print("Bot: Bye 👋")
        break

    reply = chatbot_response(msg)
    print("Bot:", reply)

Chatbot ready!
You: Hi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Bot: Hey!
You: Byee
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Bot: Hello!
You: Quit
Bot: Bye 👋
